# 04b · A second test on campaigns nobody has seen

Notebook 04 used its test campaigns (Nov/2021 to Feb/2025) once. It found a failure: almost all missed blooms were in the dry-season campaigns (Aug, Nov), and the season and temperature inputs had learned a pattern from 2016–2020 that did not hold later.

COGERH's online database already holds **six newer campaigns (May/2025 to Aug/2026)** that are in no PDF report yet. No model, threshold or choice has seen them, so they make a fair second test.

**Written down before extracting or scoring anything (see the commit time of this notebook):**
- **Model A:** the model chosen in notebook 04 (logistic regression, 15 inputs), unchanged, with its validation threshold.
- **Model B:** the same logistic regression with only what the satellite and the volume gauge see (NDCI, NDTI, MNDWI, water share, volume, volume change). It is trained on the same training campaigns, and its threshold is chosen on the same validation campaigns (catch at least 80% of blooms).
- **Baselines:** persistence (the previous lab result, also from the database) and the NDCI rule with its validation threshold.
- **Hypothesis:** B misses fewer blooms than A in the dry-season campaigns.
- **Demo rule:** the risk map uses B if its PR-AUC here is within 0.02 of A's, or better (the same "prefer the simpler model" rule as notebook 04); otherwise it keeps A.
- These campaigns are scored **once**. Nothing is tuned on them.

**Caveats:** the counts come only from the database (there is no PDF to cross-check them yet); stored volume comes from the portal's daily-level page on the sample date (in training it came from the PDF reports).

In [ ]:
import sys, time, warnings
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import PROCESSED, RAW, OUTPUTS, CYANO_THRESHOLD
from modeling import FEATURES, SAT_WATER, time_split, metrics, threshold_for_recall, best_f1_threshold
from satellite import CRS, features_for_sample
from weather import grid_cell, fetch_cell, window_features
from fetch_hidro import fetch_volumes

warnings.filterwarnings("ignore")
SEED = 42
pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
new = pd.read_csv(PROCESSED / "new_campaign_samples.csv", parse_dates=["date", "prev_date"])
res = gpd.read_file(PROCESSED / "reservoirs.geojson").set_index("reservoir")
print(len(new), "database samples in", new["campaign"].nunique(), "new campaigns,", new["reservoir"].nunique(), "reservoirs")
print("above 20,000 cells/mL:", round((new["cyano"] > CYANO_THRESHOLD).mean(), 3))

## Sentinel-2 (same rules as notebook 03; extracted with `src/run_s2.py`)

In [ ]:
aoi_utm = res.to_crs(CRS).geometry
def job(r):
    return {"reservoir": r.reservoir, "date": r.date,
            **features_for_sample(r.reservoir, r.date, aoi_utm[r.reservoir], res.geometry[r.reservoir].bounds,
                                  PROCESSED / "s2_cache", cache_only=True)}
with ThreadPoolExecutor(8) as ex:
    s2 = pd.DataFrame(list(ex.map(job, new.itertuples())))
print(s2["status"].value_counts().to_string())
t2 = new.merge(s2[s2["status"] == "ok"], on=["reservoir", "date"])
print(len(t2), "samples with a clear image")

## Weather, stored volume, season

Weather: NASA POWER, the same windows as in training. Volume: the portal's daily-level page on the sample date and on the previous sample's date (`volume_change`), one page per date, fetched one at a time and cached.

In [ ]:
res["cell"] = [grid_cell(a, b) for a, b in zip(res["lat"], res["lon"])]
series = {c: fetch_cell(*c, RAW / "power") for c in res.loc[t2["reservoir"].unique(), "cell"].unique()}
w = pd.DataFrame([window_features(series[res.at[r.reservoir, "cell"]], r.date) for r in t2.itertuples()], index=t2.index)
t2 = t2.join(w)

vols = {}
for d in sorted(set(t2["date"]) | set(t2["prev_date"].dropna())):
    vols[d] = fetch_volumes(PROCESSED.parent.parent, d).drop_duplicates("reservoir").set_index("reservoir")["volume_pct"]
    time.sleep(0.5)
t2["volume_pct"] = [vols[d].get(r, np.nan) for r, d in zip(t2["reservoir"], t2["date"])]
t2["prev_volume"] = [vols[d].get(r, np.nan) if pd.notna(d) else np.nan for r, d in zip(t2["reservoir"], t2["prev_date"])]
t2["volume_change"] = t2["volume_pct"] - t2["prev_volume"]
m = t2["date"].dt.month
t2["month_sin"], t2["month_cos"] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
t2["rainy_season"] = m.between(2, 5).astype(int)
t2["y"] = (t2["cyano"] > CYANO_THRESHOLD).astype(int)
t2["prev_log10_cyano"] = np.log10(t2["prev_cyano"].clip(lower=1))
t2["season"] = np.where(t2["campaign"].str[-2:].isin(["08", "11"]), "dry (Aug, Nov)", "wet (Feb, May)")
print(len(vols), "volume pages;", "volume found for", int(t2["volume_pct"].notna().sum()), "of", len(t2))
print(t2.groupby("campaign").agg(n=("y", "size"), blooms=("y", "sum")).T.to_string())

## Models A and B, thresholds from validation (as in notebook 04)

In [ ]:
df = pd.read_csv(PROCESSED / "features.csv", parse_dates=["date"])
df["split"] = time_split(df)
tr, va = df[df["split"] == "train"], df[df["split"] == "val"]
va_p = va[va["prev_class"].notna()]

A = joblib.load(OUTPUTS / "model.joblib")
assert A["name"] == "Logistic regression" and A["features"] == FEATURES
B_model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(),
                        LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)).fit(tr[SAT_WATER], tr["y"])
B = {"name": "Logistic regression (satellite + stored water)", "model": B_model, "features": SAT_WATER,
     "threshold": threshold_for_recall(va_p["y"], B_model.predict_proba(va_p[SAT_WATER])[:, 1], 0.80),
     "train_campaigns": sorted(tr["campaign"].unique())}
t_ndci = best_f1_threshold(va_p["y"], va_p["NDCI"].values)
print({"A threshold": round(A["threshold"], 3), "B threshold": round(B["threshold"], 3), "NDCI t": round(t_ndci, 4)})
print("validation PR-AUC: A", round(average_precision_score(va_p["y"], A["model"].predict_proba(va_p[FEATURES])[:, 1]), 3),
      "| B", round(average_precision_score(va_p["y"], B_model.predict_proba(va_p[SAT_WATER])[:, 1]), 3))

## Second test: scored once

In [ ]:
d = t2[t2["prev_cyano"].notna()].copy()
score = {"A": A["model"].predict_proba(d[FEATURES])[:, 1], "B": B_model.predict_proba(d[SAT_WATER])[:, 1]}
rows = {
    "Persistence (last lab result)": (metrics(d["y"], (d["prev_cyano"] > CYANO_THRESHOLD).astype(int), d["prev_log10_cyano"]), d["prev_log10_cyano"].values),
    "Majority class (always bloom)": (metrics(d["y"], np.ones(len(d), int), np.ones(len(d))), np.ones(len(d))),
    "NDCI > t": (metrics(d["y"], (d["NDCI"] > t_ndci).astype(int), d["NDCI"]), d["NDCI"].values),
    "A: logistic regression, 15 inputs": (metrics(d["y"], (score["A"] >= A["threshold"]).astype(int), score["A"]), score["A"]),
    "B: satellite + stored water, 6 inputs": (metrics(d["y"], (score["B"] >= B["threshold"]).astype(int), score["B"]), score["B"]),
}
results2 = pd.DataFrame({n: r for n, (r, _) in rows.items()}).T
results2["PR_AUC_clean"] = pd.Series({n: average_precision_score(1 - d["y"], -s) for n, (_, s) in rows.items()})
results2.to_csv(OUTPUTS / "test2_results.csv")
print(f"chance level: {d['y'].mean():.3f} (blooms), {1 - d['y'].mean():.3f} (clean)")
results2[["n", "positives", "recall", "precision", "F1", "PR_AUC", "PR_AUC_clean", "balanced_acc", "missed_blooms (FN)", "false_alarms (FP)"]]

In [ ]:
rng = np.random.default_rng(SEED)
y, pers = d["y"].values, d["prev_log10_cyano"].values
idx = [i for i in (rng.integers(0, len(y), len(y)) for _ in range(1000)) if y[i].min() != y[i].max()]
for label, s in [("A", score["A"]), ("B", score["B"]), ("NDCI alone", d["NDCI"].values)]:
    diffs = [average_precision_score(y[i], s[i]) - average_precision_score(y[i], pers[i]) for i in idx]
    lo, hi = np.percentile(diffs, [2.5, 97.5])
    print(f"PR-AUC({label}) - PR-AUC(persistence): mean {np.mean(diffs):+.3f}, 95% interval [{lo:+.3f}, {hi:+.3f}]")

## The hypothesis: dry-season campaigns

In [ ]:
rows = []
for season, g in d.groupby("season"):
    for name, key in [("A", "A"), ("B", "B")]:
        s = score[key][d.index.get_indexer(g.index)]
        mdl = A if key == "A" else B
        r = metrics(g["y"], (s >= mdl["threshold"]).astype(int), s)
        rows.append({"season": season, "model": name, "n": r["n"], "blooms": r["positives"], "recall": r["recall"],
                     "missed_blooms": r["missed_blooms (FN)"], "false_alarms": r["false_alarms (FP)"]})
    p = metrics(g["y"], (g["prev_cyano"] > CYANO_THRESHOLD).astype(int))
    rows.append({"season": season, "model": "persistence", "n": p["n"], "blooms": p["positives"], "recall": p["recall"],
                 "missed_blooms": p["missed_blooms (FN)"], "false_alarms": p["false_alarms (FP)"]})
pd.DataFrame(rows).set_index(["season", "model"]).round(3)

## Demo rule (fixed before scoring)

In [ ]:
ap_A, ap_B = results2.loc["A: logistic regression, 15 inputs", "PR_AUC"], results2.loc["B: satellite + stored water, 6 inputs", "PR_AUC"]
demo = B if ap_B >= ap_A - 0.02 else A
print(f"PR-AUC A {ap_A:.3f}, B {ap_B:.3f} -> the risk map uses: {demo['name']}")
joblib.dump(demo, OUTPUTS / "model_demo.joblib")